# RAG Workflow Test
Mirrors the workflow used in `src/app.py`: load embeddings, connect to the vector store, build the RAG chain, and run queries with multi-turn chat history.

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

# Make the src package importable from the project root
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))

Project root: /home/daniel/Estudos e projetos/Repos/NLPChatbot
ANTHROPIC_API_KEY set: True


## 2. Load Embedding Model

In [3]:
from src.ingest import get_embeddings

print("Loading embedding model (this may take a moment on first run)...")
embeddings = get_embeddings()
print("Embedding model loaded.")

Loading embedding model (this may take a moment on first run)...
Embedding model loaded.


## 3. Load Vector Store

Loads the existing Chroma store. If it does not exist yet, run ingestion first.

In [4]:
from src.config import CHROMA_DB_DIR
from src.ingest import load_vectorstore, run_ingestion

if CHROMA_DB_DIR.exists():
    print(f"Found existing vector store at {CHROMA_DB_DIR}. Loading...")
    vectorstore = load_vectorstore(embeddings)
    print("Vector store loaded.")
else:
    print("No vector store found. Running ingestion...")
    vectorstore = run_ingestion()
    print("Ingestion complete.")

Found existing vector store at /home/daniel/Estudos e projetos/Repos/NLPChatbot/chroma_db. Loading...
Vector store loaded.


In [5]:
# Sanity-check: how many documents are in the collection?
count = vectorstore._collection.count()
print(f"Documents in collection: {count}")

Documents in collection: 1006


## 4. Build RAG Chain

In [6]:
from src.rag_chain import build_rag_chain

rag_chain, retriever = build_rag_chain(vectorstore)
print("RAG chain and retriever ready.")

RAG chain and retriever ready.


## 5. Single-Turn Query

In [7]:
question = "What is a language model?"

source_docs = retriever.invoke(question)
answer = rag_chain.invoke({"question": question, "chat_history": []})

print("Question:", question)
print("\nAnswer:")
print(answer)

Question: What is a language model?

Answer:
Based on the provided references, a language model is **a computational system that can predict the next word from previous words**. [Source: 7.pdf, Page: 1]

More specifically, language models **offer a way to assign a probability to a sentence or other sequence of words or tokens, and to predict a word or token from preceding words or tokens**. [Source: 3.pdf, Page: 20]

In other words, given a context or prefix of words, a language model assigns probabilities to what word might come next in the sequence.


In [8]:
# Inspect retrieved source chunks
print(f"Retrieved {len(source_docs)} chunks:\n")
for i, doc in enumerate(source_docs, 1):
    filename = os.path.basename(doc.metadata.get("source", "unknown"))
    page = doc.metadata.get("page", "?")
    print(f"[{i}] {filename} — Page {page}")
    print(doc.page_content[:300] + ("..." if len(doc.page_content) > 300 else ""))
    print()

Retrieved 5 chunks:

[1] 3.pdf — Page 20
that allows us to introduce many of the basic concepts in language modeling.
• Language models offer a way to assign a probability to a sentence or other
sequence of words or tokens, and to predict a word or token from preceding
words or tokens.
• N-grams are perhaps the simplest kind of language mo...

[2] 7.pdf — Page 26
HISTORICAL NOTES 27
The technology used for language models can also be applied to other domains
and tasks, like vision, speech, and genetics. The term foundation model is some-foundation
model
times used as a more general term for this use of large language model technology
across domains and areas...

[3] 7.pdf — Page 1
The room wasn’t just big it was enormous
The square root of 4 is 2
The author of “A Room of One’s Own” is Virginia Woolf
The professor said that he
From the ﬁrst sentence a model can learn ontological facts like that roses and
dahlias and peonies are all kinds of ﬂowers. From the second, a model cou...

[4] 7

## 6. Multi-Turn Chat (with History)

In [9]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def chat(question: str) -> str:
    answer = rag_chain.invoke({"question": question, "chat_history": chat_history})
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    return answer


# Turn 1
q1 = "What is the difference between unigram and bigram language models?"
a1 = chat(q1)
print("Q1:", q1)
print("A1:", a1)

Q1: What is the difference between unigram and bigram language models?
A1: Based on the provided references, here are the key differences between unigram and bigram language models:

**1. Context Consideration:**
- **Unigram models**: Do not consider any context or word relationships. They treat each word independently.
- **Bigram models**: Consider the previous word as context when predicting the next word [Source: 3.pdf, Page: 9]

**2. Probability Calculation:**
The perplexity formulas illustrate this difference:
- **Bigram perplexity**: Uses conditional probabilities P(wi|wi-1), meaning the probability of a word given the previous word [Source: 3.pdf, Page: 9]
- The bigram probability calculation shows this dependency, such as P(want|i) or P(english|want) [Source: 3.pdf, Page: 5]

**3. Coherence of Generated Text:**
When generating random sentences:
- **Unigram sentences**: "show no coherent relation between words nor any sentence-final punctuation"
- **Bigram sentences**: "have som

In [10]:
# Turn 2 — follow-up that relies on chat history
q2 = "Can you give an example of each?"
a2 = chat(q2)
print("Q2:", q2)
print("A2:", a2)

Q2: Can you give an example of each?
A2: Based on the provided references, here are examples of unigram and bigram language models:

## Unigram Example:

**Random sentence generated from unigram probabilities:**
"fifth, an, of, futures, the, an, incorporated, a, a, the, inflation, most, dollars, quarter, in, is, mass" [Source: 3.pdf, Page: 11]

As noted, this shows "no coherent relation between words" [Source: 3.pdf, Page: 11].

## Bigram Example:

**Random sentence generated from bigram probabilities:**
"Outside new cars and light trucks are being made in the fourthquarter sales in the U.S. could be sold" [Source: 3.pdf, Page: 11]

This demonstrates "some local word-to-word coherence" [Source: 3.pdf, Page: 11].

**Bigram probability calculations:**
The references show specific bigram probabilities like:
- P(i|`<s>`) = 0.25 (probability of "i" starting a sentence)
- P(want|i) = 0.33 (probability of "want" following "i")
- P(english|want) = 0.0011 (probability of "english" following "wa

In [11]:
# Turn 3
q3 = "Which one is more commonly used in practice and why?"
a3 = chat(q3)
print("Q3:", q3)
print("A3:", a3)

Q3: Which one is more commonly used in practice and why?
A3: I don't know based on the provided references.

The references provided discuss the differences between unigram and bigram models and give examples of each, but they do not contain information about which one is more commonly used in modern NLP practice or the reasons why one might be preferred over the other in practical applications.


In [12]:
# Print the full conversation
print("=== Full conversation ===")
for msg in chat_history:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"\n[{role}]")
    print(msg.content)

=== Full conversation ===

[User]
What is the difference between unigram and bigram language models?

[Assistant]
Based on the provided references, here are the key differences between unigram and bigram language models:

**1. Context Consideration:**
- **Unigram models**: Do not consider any context or word relationships. They treat each word independently.
- **Bigram models**: Consider the previous word as context when predicting the next word [Source: 3.pdf, Page: 9]

**2. Probability Calculation:**
The perplexity formulas illustrate this difference:
- **Bigram perplexity**: Uses conditional probabilities P(wi|wi-1), meaning the probability of a word given the previous word [Source: 3.pdf, Page: 9]
- The bigram probability calculation shows this dependency, such as P(want|i) or P(english|want) [Source: 3.pdf, Page: 5]

**3. Coherence of Generated Text:**
When generating random sentences:
- **Unigram sentences**: "show no coherent relation between words nor any sentence-final punctua